# Automation and CLI Tools

Python excels at automating repetitive tasks and building command-line tools.

This notebook covers:
- **`os` and `shutil`** — file system operations
- **`subprocess`** — run shell commands
- **`csv`** — read and write CSV files
- **`json`** — work with JSON config files
- **`argparse`** — parse command-line arguments
- **`click`** — build beautiful CLI tools

## 1. File Operations with `os` and `shutil`

In [ ]:
import os
import shutil

# Create directories
os.makedirs('/tmp/demo/subdir1', exist_ok=True)
os.makedirs('/tmp/demo/subdir2', exist_ok=True)

# Create test files
for i in range(5):
    with open(f'/tmp/demo/file{i}.txt', 'w') as f:
        f.write(f'Content {i}')

# List directory
print('Directory contents:')
for item in os.listdir('/tmp/demo'):
    full_path = os.path.join('/tmp/demo', item)
    is_dir = os.path.isdir(full_path)
    print(f'  {"📁" if is_dir else "📄"} {item}')

In [ ]:
import os, shutil

# os.walk — recursively traverse directories
print('Walking /tmp/demo:')
for root, dirs, files in os.walk('/tmp/demo'):
    level = root.replace('/tmp/demo', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        print(f'{indent}  {f}')

# Copy, move, delete
shutil.copy('/tmp/demo/file0.txt', '/tmp/demo/subdir1/file0_copy.txt')
shutil.move('/tmp/demo/file1.txt', '/tmp/demo/subdir2/file1_moved.txt')

print('\nAfter operations:')
for root, dirs, files in os.walk('/tmp/demo'):
    for f in files:
        print(f'  {os.path.join(root, f).replace("/tmp/demo/", "")}')

## 2. `subprocess` — Run Shell Commands

In [ ]:
import subprocess

# Run a simple command and capture output
result = subprocess.run(
    ['python', '--version'],
    capture_output=True,
    text=True  # return strings instead of bytes
)
print(f'Python version: {result.stdout.strip()}')
print(f'Return code: {result.returncode}')

In [ ]:
import subprocess

# List files (cross-platform)
cmd = ['dir'] if os.name == 'nt' else ['ls', '-la']
result = subprocess.run(cmd, capture_output=True, text=True, cwd='/tmp')
if result.returncode == 0:
    print(result.stdout[:300])
else:
    print(f'Error: {result.stderr[:100]}')

## 3. CSV Files

In [ ]:
import csv

# Write CSV
employees = [
    {'name': 'Alice', 'department': 'Engineering', 'salary': 80000},
    {'name': 'Bob', 'department': 'Marketing', 'salary': 65000},
    {'name': 'Charlie', 'department': 'Engineering', 'salary': 90000},
]

with open('/tmp/employees.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['name', 'department', 'salary'])
    writer.writeheader()
    writer.writerows(employees)

# Read CSV
with open('/tmp/employees.csv', 'r') as f:
    reader = csv.DictReader(f)
    loaded = list(reader)

print(f'Loaded {len(loaded)} employees:')
for emp in loaded:
    print(f"  {emp['name']:10} {emp['department']:15} £{int(emp['salary']):,}")

## 4. `argparse` — Command-Line Arguments

In [ ]:
# argparse is used when running scripts from the terminal
# Shown here as what you'd put in a script file

argparse_example = '''
# script.py
import argparse

def main():
    parser = argparse.ArgumentParser(
        description=\'Process a CSV file and generate a report.\'
    )
    
    # Positional argument (required)
    parser.add_argument(\'input\', type=str, help=\'Input CSV file path\')
    
    # Optional arguments
    parser.add_argument(\'--output\', \'--o\', type=str, default=\'report.txt\',
                        help=\'Output file (default: report.txt)\')
    parser.add_argument(\'--format\', choices=[\'text\', \'json\', \'html\'],
                        default=\'text\', help=\'Report format\')
    parser.add_argument(\'--verbose\', \'-v\', action=\'store_true\',
                        help=\'Enable verbose output\')
    parser.add_argument(\'--limit\', type=int, default=None,
                        help=\'Max rows to process\')
    
    args = parser.parse_args()
    
    if args.verbose:
        print(f\'Processing {args.input}...\')
    
    # Process args.input, write to args.output in args.format
    print(f\'Report written to {args.output}\')

if __name__ == \'__main__\':
    main()
'''

# Usage: python script.py data.csv --output report.json --format json -v
print('Example argparse script:')
print(argparse_example)

## 5. `click` — Beautiful CLI Tools

**click** makes it easy to build powerful CLI tools with decorators.  
Install: `pip install click`

In [ ]:
click_example = '''
# cli.py
import click
import csv
import json

@click.group()
@click.version_option(\'1.0.0\')
def cli():
    \'\'\'CSV processing tool.\'\'\'  # This becomes the --help text
    pass


@cli.command()
@click.argument(\'input_file\', type=click.Path(exists=True))
@click.option(\'--output\', \'-o\', default=\'output.csv\', help=\'Output file\')
@click.option(\'--separator\', \'-s\', default=\',\', help=\'Field separator\')
@click.option(\'--verbose\', \'-v\', is_flag=True, help=\'Verbose output\')
def process(input_file, output, separator, verbose):
    \'\'\'Process a CSV file.\'\'\'
    if verbose:
        click.echo(f\'Processing {input_file}...\')
    
    with open(input_file) as f:
        rows = list(csv.DictReader(f, delimiter=separator))
    
    click.echo(f\'Processed {len(rows)} rows\', err=verbose)
    
    with open(output, \'w\') as f:
        if rows:
            writer = csv.DictWriter(f, fieldnames=rows[0].keys())
            writer.writeheader()
            writer.writerows(rows)
    
    click.secho(f\'Done! Written to {output}\', fg=\'green\')


@cli.command()
@click.argument(\'csv_file\', type=click.Path(exists=True))
@click.option(\'--column\', \'-c\', help=\'Column to summarise\')
def summary(csv_file, column):
    \'\'\'Show summary statistics for a CSV file.\'\'\'
    with open(csv_file) as f:
        rows = list(csv.DictReader(f))
    
    click.echo(f\'Rows: {len(rows)}\')
    click.echo(f\'Columns: {list(rows[0].keys()) if rows else []}\')


if __name__ == \'__main__\':
    cli()
'''

print('Click CLI example:')
print(click_example[:800], '...')

## Mini-Project: CSV Report Generator CLI

In [ ]:
import csv
import json
import os
from collections import defaultdict

# Sample sales CSV data
sales_data = '''
date,region,product,quantity,unit_price
2024-01-05,North,Laptop,2,999.99
2024-01-07,South,Mouse,10,29.99
2024-01-08,North,Keyboard,5,79.99
2024-01-10,East,Laptop,1,999.99
2024-01-12,West,Mouse,8,29.99
2024-01-15,South,Laptop,3,999.99
2024-01-18,North,Monitor,2,399.99
2024-01-20,East,Keyboard,7,79.99
2024-01-22,West,Monitor,1,399.99
2024-01-25,South,Keyboard,4,79.99
'''.strip()

# Write sample data
with open('/tmp/sales.csv', 'w') as f:
    f.write(sales_data)


def analyze_csv(filepath, group_by='product', value_col='quantity', calc='sum'):
    """
    Analyze a CSV file and generate a report.
    
    Args:
        filepath: Path to CSV file
        group_by: Column to group by
        value_col: Column to aggregate
        calc: 'sum', 'avg', 'count', 'max', 'min'
    """
    with open(filepath) as f:
        rows = list(csv.DictReader(f))
    
    if not rows:
        print('No data')
        return
    
    # Group data
    groups = defaultdict(list)
    for row in rows:
        key = row.get(group_by, 'Unknown')
        try:
            val = float(row.get(value_col, 0))
        except ValueError:
            val = 0
        groups[key].append(val)
    
    # Aggregate
    results = {}
    for key, values in groups.items():
        if calc == 'sum':   results[key] = sum(values)
        elif calc == 'avg': results[key] = sum(values) / len(values)
        elif calc == 'count': results[key] = len(values)
        elif calc == 'max': results[key] = max(values)
        elif calc == 'min': results[key] = min(values)
    
    return results


def print_report(results, title, currency=False):
    print(f'\n=== {title} ===')
    sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)
    max_val = max(results.values())
    
    for key, val in sorted_results:
        bar = '█' * int(val / max_val * 20)
        prefix = '£' if currency else ''
        print(f'{key:12} {prefix}{val:8.2f}  {bar}')


# Revenue = quantity × price
with open('/tmp/sales.csv') as f:
    rows = list(csv.DictReader(f))

for row in rows:
    row['revenue'] = str(float(row['quantity']) * float(row['unit_price']))

# Save enhanced CSV
with open('/tmp/sales_enhanced.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

# Generate reports
revenue_by_product = analyze_csv('/tmp/sales_enhanced.csv', 'product', 'revenue', 'sum')
revenue_by_region = analyze_csv('/tmp/sales_enhanced.csv', 'region', 'revenue', 'sum')
qty_by_product = analyze_csv('/tmp/sales_enhanced.csv', 'product', 'quantity', 'sum')

print_report(revenue_by_product, 'Revenue by Product', currency=True)
print_report(revenue_by_region, 'Revenue by Region', currency=True)
print_report(qty_by_product, 'Units Sold by Product')

# Save JSON report
report = {
    'total_revenue': sum(revenue_by_product.values()),
    'total_units': sum(qty_by_product.values()),
    'by_product': revenue_by_product,
    'by_region': revenue_by_region,
}
with open('/tmp/sales_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print(f'\n=== SUMMARY ===')
print(f'Total Revenue: £{report["total_revenue"]:,.2f}')
print(f'Total Units: {report["total_units"]:.0f}')
print('Full report saved to /tmp/sales_report.json')